# 01 – Recogida de Datos (Data Collection)

**Responsable**: David (Data Analyst)  
**Fecha**: Mayo 2026  
**Objetivo**: Obtener ofertas de empleo del sector datos en España para complementar los datasets estáticos proporcionados por el bootcamp.

---

## Datasets ya disponibles

El proyecto cuenta con tres fuentes de datos iniciales:

1. `tecnoempleo_spain_2026.csv` → 600 ofertas locales, salario (78 % nulos), ubicación.
2. `data_science_job_posts_2025.csv` → 944 ofertas internacionales (143 en España), skills y salario sin nulos.
3. `stackoverflow_2025_results.csv` → Encuesta de Stack Overflow para análisis demográfico y tecnológico.

Estos datasets son la base del EDA, pero necesitábamos más volumen de ofertas españolas recientes.

---

## Evaluación de fuentes externas

Se probaron varias técnicas de scraping sobre los principales portales de empleo en España. La tabla resume los resultados:

| Portal      | Método                     | Resultado                        |
|-------------|----------------------------|----------------------------------|
| InfoJobs    | requests + BeautifulSoup   | CAPTCHA                          |
| InfoJobs    | Playwright (navegador real)| Bloqueo Distil Networks          |
| Indeed      | requests + BeautifulSoup   | 403 Security Check               |
| Indeed      | Playwright visible (panel) | Cloudflare Turnstile (no superado)|
| Indeed      | Playwright + clics + pausa | Cloudflare + cierre de contexto  |
| InfoJobs API| Registro de app            | Registro cerrado temporalmente   |

**Conclusión**: Las protecciones anti‑bot de estos portales impiden el scraping ético y automatizado. La API pública de InfoJobs era la vía oficial, pero el registro de nuevas aplicaciones se encuentra cerrado.

---

## Solución final: API de Adzuna

Se identificó la API pública de [Adzuna](https://developer.adzuna.com) como alternativa viable:

- **Capa gratuita**: 1000 peticiones/mes, suficiente para el proyecto.
- **Autenticación simple**: `app_id` + `app_key` como parámetros de URL.
- **Datos devueltos**: título, empresa, ubicación, descripción completa, salario (cuando está disponible), tipo de contrato y URL original.
- **Paginación nativa**: permite recorrer hasta 5 páginas de 50 resultados cada una.
- **Sin bloqueos**: no se detectaron CAPTCHAs ni restricciones durante las pruebas.

Gracias a esta API podemos extraer cientos de ofertas españolas con su descripción completa, enriqueciendo enormemente el dataset final.

---

## Seguridad de las credenciales

Las claves de la API **nunca** se escriben en el código. Se almacenan en un archivo `.env` (ignorado por Git) y se cargan en tiempo de ejecución con `python-dotenv`.

Para que cualquier miembro del equipo pueda replicar la extracción, se ha creado un archivo `.env.example` **sin valores reales** que sirve como plantilla. Cada persona debe copiarlo a `.env` y añadir sus propias credenciales de Adzuna.

### Instrucciones para el equipo
1. Copia `.env.example` y renómbralo a `.env`.
2. Regístrate en [Adzuna Developer](https://developer.adzuna.com) y obtén tu `APP_ID` y `APP_KEY`.
3. Edita `.env` con tus claves.

---

## Guía de uso para ejecutar el scraping

Esta sección está pensada para que cualquier persona del equipo, incluso sin experiencia previa, pueda lanzar la extracción de ofertas de empleo.

### ¿Qué hace este notebook?
- Se conecta a la API pública de Adzuna, un portal de empleo.
- Busca ofertas usando palabras clave (por ejemplo, "data scientist") y ubicaciones (Madrid, Barcelona…).
- Guarda la información (título, empresa, descripción, salario…) en un archivo CSV dentro de la carpeta `data/raw/`.

### Requisitos previos

1. **Tener Python y Conda instalados** (ya lo tienes si estás en el bootcamp).
2. **Instalar las librerías necesarias**. Abre un terminal en la carpeta del proyecto y ejecuta:
   ```bash
   pip install -r requirements.txt
   ```
   Esto instalará `requests`, `pandas`, `python-dotenv` y el resto de dependencias.

3. **Obtener credenciales de Adzuna** (gratuitas):
   - Ve a [https://developer.adzuna.com/signup](https://developer.adzuna.com/signup) y crea una cuenta.
   - Dentro del panel, verás un `Application ID` y un `Application Key`. Cópialos.

4. **Crear tu archivo `.env`**:
   - En la carpeta raíz del proyecto hay un archivo llamado `.env.example`. Cópialo y renómbralo como `.env`.
   - Ábrelo con un editor de texto y reemplaza los valores de ejemplo:
     ```
     ADZUNA_APP_ID=tu_app_id_real
     ADZUNA_APP_KEY=tu_app_key_real
     ```
   - **Importante**: el archivo `.env` nunca se sube a GitHub (ya está incluido en `.gitignore`). Cada persona debe crear el suyo.

### Cómo ejecutar el scraping

Una vez configurado el entorno, simplemente ejecuta las celdas en orden:

1. **Celda de importaciones** – Carga las librerías y las claves desde `.env`.
2. **Celda de la función** – Define la lógica de extracción (no necesitas modificarla).
3. **Celda de prueba** – Realiza una búsqueda pequeña para verificar que todo funciona.
4. **Celda de búsquedas masivas** – Lanza todas las combinaciones de roles y ciudades para llenar el dataset final.

Si todo va bien, al terminar tendrás un archivo `data/raw/scraped_jobs.csv` con cientos de ofertas listas para el análisis.

### Solución de problemas frecuentes

- **Error "Falta ADZUNA_APP_ID..."** → Revisa que el archivo `.env` existe y contiene las claves correctas.
- **Error 401 o 403** → Las credenciales no son válidas. Regenera las claves en el panel de Adzuna.
- **"No hay más resultados en página X"** → Es normal: la API solo devuelve una cantidad limitada de ofertas por búsqueda. Puedes probar con otras palabras clave o ciudades.
- **Límite de peticiones agotado** → La cuenta gratuita permite 1000 peticiones al mes. Si se supera, espera hasta el mes siguiente o contacta con Adzuna para ampliar el límite.

Si tienes cualquier duda, consulta al equipo en Discord.

---

## Importaciones y carga de variables

In [1]:
import os
import requests
import time
import pandas as pd
from dotenv import load_dotenv

# Cargar variables desde .env
load_dotenv()

APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")

if not APP_ID or not APP_KEY:
    raise ValueError("Falta ADZUNA_APP_ID o ADZUNA_APP_KEY en el archivo .env. Revisa .env.example para más información.")

# Rutas del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DEFAULT_OUTPUT = "scraped_jobs.csv"

## Función principal scrape_adzuna()

In [2]:
def scrape_adzuna(keyword, location, output_file=DEFAULT_OUTPUT, max_pages=5):
    """
    Busca ofertas en España usando la API de Adzuna y las guarda en un CSV unificado.

    Parámetros
    ----------
    keyword : str
        Palabras clave para la búsqueda (ej. "data scientist").
    location : str
        Ubicación (ej. "Madrid", "Barcelona", "remoto").
    output_file : str
        Nombre del archivo CSV en data/raw/.
    max_pages : int
        Número máximo de páginas a recorrer (cada página devuelve hasta 50 resultados).

    Retorna
    -------
    pd.DataFrame con todas las ofertas únicas acumuladas.
    """
    base_url = "https://api.adzuna.com/v1/api/jobs/es/search"
    all_offers = []

    for page in range(1, max_pages + 1):
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "what": keyword,
            "where": location,
            "content-type": "application/json",
            "results_per_page": 50
        }
        resp = requests.get(f"{base_url}/{page}", params=params)

        if resp.status_code != 200:
            print(f"  [!] Error en página {page}: {resp.status_code}")
            break

        data = resp.json()
        offers = data.get("results", [])
        if not offers:
            print(f"  [i] No hay más resultados en página {page}. Finalizando.")
            break

        for job in offers:
            all_offers.append({
                "titulo": job.get("title"),
                "empresa": job.get("company", {}).get("display_name"),
                "ubicacion": job.get("location", {}).get("display_name"),
                "enlace": job.get("redirect_url"),
                "metadatos": f"{job.get('contract_type', '')} | {job.get('category', {}).get('label', '')}",
                "descripcion": job.get("description"),
                "salario_min": job.get("salary_min"),
                "salario_max": job.get("salary_max"),
                "salario_moneda": job.get("salary_currency", "EUR")
            })

        print(f"  Página {page}: {len(offers)} ofertas extraídas. Total acumulado: {len(all_offers)}")
        time.sleep(1.1)  # Respetar límite de peticiones

    if not all_offers:
        print("No se obtuvieron ofertas.")
        return pd.DataFrame()

    df_nuevas = pd.DataFrame(all_offers)
    print(f"Total nuevas ofertas: {len(df_nuevas)}")

    # Gestión del archivo unificado
    raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
    os.makedirs(raw_dir, exist_ok=True)
    filepath = os.path.join(raw_dir, output_file)

    if os.path.exists(filepath):
        df_existente = pd.read_csv(filepath)
        print(f"Ofertas ya existentes en {output_file}: {len(df_existente)}")
        df_total = pd.concat([df_existente, df_nuevas], ignore_index=True)
        df_total = df_total.drop_duplicates(subset=["enlace"], keep="first")
    else:
        df_total = df_nuevas

    df_total.to_csv(filepath, index=False, encoding="utf-8")
    print(f"Total de ofertas únicas guardadas en {filepath}: {len(df_total)}")
    return df_total

## Prueba rápida

In [3]:
# Prueba con una sola búsqueda
df_test = scrape_adzuna("data scientist", "Madrid", max_pages=2)
df_test[["titulo", "empresa", "descripcion"]].head()

  Página 1: 29 ofertas extraídas. Total acumulado: 29
  [i] No hay más resultados en página 2. Finalizando.
Total nuevas ofertas: 29
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 29


,titulo,empresa,descripcion
0,Data Scientist,Abound,About Abound We’re redefining consumer lending...
1,Data Scientist,Encore Capital Group,The role is embedded day‑to‑day with the Spani...
2,Data Scientist,Vodafone,Build and deliver data science & gen AI soluti...
3,Data Scientist,Mática Partners,¿Te apasiona el BigData? ¿Quieres formar parte...
4,DATA SCIENTIST ASSOCIATE,BBVA,· · Área: DATA Inscríbete hasta: 16/02/2023 So...


## Búsquedas masivas

In [4]:
# Definición de roles y ciudades
roles = [
    "data scientist",
    "data engineer",
    "data analyst",
    "machine learning engineer",
    "business intelligence",
    "big data",
    "analista de datos"
]

ciudades = [
    "Madrid",
    "Barcelona",
    "Valencia",
    "Bilbao",
    "Sevilla",
    "Zaragoza",
    "Malaga",
    "remoto"
]

# Ejecutar todas las combinaciones (56 búsquedas)
for rol in roles:
    for ciudad in ciudades:
        print(f"\n{'='*50}")
        print(f"Buscando: {rol} en {ciudad}")
        scrape_adzuna(rol, ciudad, max_pages=1)  # max_pages=1 para no agotar el cupo
        print(f"Finalizado: {rol} en {ciudad}")

print("\nTodas las búsquedas completadas. Datos guardados en data/raw/scraped_jobs.csv")


Buscando: data scientist en Madrid
  Página 1: 29 ofertas extraídas. Total acumulado: 29
Total nuevas ofertas: 29
Ofertas ya existentes en scraped_jobs.csv: 29
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 29
Finalizado: data scientist en Madrid

Buscando: data scientist en Barcelona
  Página 1: 46 ofertas extraídas. Total acumulado: 46
Total nuevas ofertas: 46
Ofertas ya existentes en scraped_jobs.csv: 29
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo2/1_Proyecto/proyecto-eda-roles-datos/data/raw/scraped_jobs.csv: 75
Finalizado: data scientist en Barcelona

Buscando: data scientist en Valencia
  Página 1: 6 ofertas extraídas. Total acumulado: 6
Total nuevas ofertas: 6
Ofertas ya existentes en scraped_jobs.csv: 75
Total de ofertas únicas guardadas en /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyec

## Lecciones aprendidas y sesgos

- **Scraping ético**: los bloqueos de Indeed e InfoJobs demuestran que las técnicas automatizadas contra portales con protecciones robustas no son viables en un contexto profesional sin autorización expresa.
- **API pública**: la vía correcta para obtener datos estructurados y sin infringir términos de uso es una API oficial.
- **Limitaciones de Adzuna**: muchas ofertas no publican salario (campos `salary_min`/`salary_max` nulos). Además, solo se ha extraído la primera página de resultados por búsqueda, por lo que podría existir un sesgo de frescura (ofertas más recientes).
- **Cobertura geográfica**: se han incluido las principales ciudades y la opción "remoto", pero localidades más pequeñas o áreas rurales quedan infrarrepresentadas.
- **Datos complementarios**: los otros datasets (Tecnoempleo, dataset internacional) compensan estas carencias al aportar skills explícitos, salario y sector industrial.

## Conclusión

Tras un exhaustivo proceso de prueba y error, la API de Adzuna se ha consolidado como la fuente de scraping principal. El dataset final unificado (`scraped_jobs.csv`) se entrega listo para la fase de limpieza, con descripciones completas y metadatos enriquecidos.